# 记忆

[记忆](https://docs.langchain.com/oss/python/langgraph/add-memory)（Memory）是一个可选模块。如非必要，你无需向智能体添加 Memory 模块。因为 StateGraph 本身就含有历史消息列表 `messages`，足以满足最基础的“记忆”需求。

需要添加 Memory 模块的情况包括：

1. 历史消息太多，需要用外部工具存储记忆
2. 触发人工干预（[interrupt](https://docs.langchain.com/oss/python/langgraph/interrupts)），需要临时保存 Agent 的状态
3. 跨对话提取用户偏好 等等

LangGraph 将记忆分为：

- [短期记忆](https://docs.langchain.com/oss/python/langchain/short-term-memory)（MemorySaver）
- [长期记忆](https://docs.langchain.com/oss/python/langchain/long-term-memory)（MemoryStore）

此外，还有一个 [LangMem](https://langchain-ai.github.io/langmem/) 也提供记忆存取功能。

In [ ]:
import os
import sqlite3

from dataclasses import dataclass
from typing_extensions import TypedDict
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

# Load shared project config from the repository-root .env
from pathlib import Path
import sys


def find_dive_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "dive_config.py").exists():
            return candidate
        nested = candidate / "LangChain" / "projects" / "dive-into-langgraph"
        if (nested / "dive_config.py").exists():
            return nested
    raise RuntimeError(
        "Start this notebook from the repository root or the "
        "dive-into-langgraph project directory."
    )


PROJECT_ROOT = find_dive_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dive_config import create_chat_model, create_embeddings, get_model_config, load_project_env

load_project_env(PROJECT_ROOT)
model_config = get_model_config()

# 加载模型
model = create_chat_model(temperature=0.7)

# 创建助手节点
def assistant(state: MessagesState):
    return {'messages': [model.invoke(state['messages'])]}


## 一、短期记忆

短期记忆（工作记忆）一般用于临时存储 Agent 或 工作流 的状态，以便在失败或重试后恢复。

### 1）在工作流中使用短期记忆

如果为工作流配置了检查点，下次调用该工作流时，会接着上一次对话内容继续聊下去。如果没有配置，将不会保留历史对话。

In [ ]:
# 创建短期记忆
checkpointer = InMemorySaver()

# 创建图
builder = StateGraph(MessagesState)

# 添加节点
builder.add_node('assistant', assistant)

# 添加边
builder.add_edge(START, 'assistant')
builder.add_edge('assistant', END)

# 使用检查点
graph = builder.compile(checkpointer=checkpointer)

## 如果不使用检查点，看看会发生什么？ 
# graph = builder.compile()

# 告诉智能体我是谁
result = graph.invoke(
    {'messages': ['你好！我是派大星']},
    {"configurable": {"thread_id": "1"}},
)

for message in result['messages']:
    message.pretty_print()


In [ ]:
# 让智能体说出我的名字
result = graph.invoke(
    {"messages": [{"role": "user", "content": "请问我是谁？"}]},
    {"configurable": {"thread_id": "1"}},  
)

for message in result['messages']:
    message.pretty_print()


### 2）在智能体中使用短期记忆

在智能体中使用短期记忆的效果，和工作流中类似。

In [ ]:
# 创建短期记忆
checkpointer = InMemorySaver()

agent = create_agent(
    model=model,
    checkpointer=checkpointer
)

# 告诉智能体我是章鱼哥
result = agent.invoke(
    {'messages': ['哈喽！我是章鱼哥']},
    {"configurable": {"thread_id": "2"}},
)

for message in result['messages']:
    message.pretty_print()


In [ ]:
# 让智能体说出我的名字
result = agent.invoke(
    {"messages": [{"role": "user", "content": "我是谁？"}]},
    {"configurable": {"thread_id": "2"}},  
)

for message in result['messages']:
    message.pretty_print()


为了验证 InMemorySaver 是否真的有效，可以将 checkpointer 注释后，再观察智能体的行为。

### 3）使用数据库保存短期记忆

如果用 SQLite 保存工作状态，即使退出程序，应该也能恢复退出之前的状态。下面我们来验证这一点。在此之前，需要安装一个 Python 包以支持 SqliteSaver 检查点：

```bash
pip install langgraph-checkpoint-sqlite
```

In [ ]:
# 删除SQLite数据库
if os.path.exists("short-memory.db"):
    os.remove("short-memory.db")


In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver

# 创建sqlite支持的短期记忆
checkpointer = SqliteSaver(
    sqlite3.connect("short-memory.db", check_same_thread=False)
)

# 创建Agent
agent = create_agent(
    model=model,
    checkpointer=checkpointer,
)

# 告诉智能体我是沙悟净
result = agent.invoke(
    {'messages': ['嗨！我是沙悟净']},
    {"configurable": {"thread_id": "3"}},
)

for message in result['messages']:
    message.pretty_print()


创建一个新的 Agent，并为它配置 SQLite 检查点。看看智能体能否从 SQLite 中读取关于我名字的记忆。

In [ ]:
# 创建一个新的Agent
new_agent = create_agent(
    model=model,
    checkpointer=checkpointer,
)

# 让智能体回忆我的名字
result = new_agent.invoke(
    {'messages': ['我是谁？']},
    {"configurable": {"thread_id": "3"}},
)

for message in result['messages']:
    message.pretty_print()


## 二、长期记忆

长期记忆一般用于保存与业务相关的重要信息，比如用户属性、流量参数等。

### 1）创建 Embedding 生成函数

长期记忆支持使用 Embedding 检索语义相近的内容。下面创建一个 Embedding 生成函数，该函数可生成检索所需的 Embedding。

In [ ]:
# Embedding dimensions
EMBED_DIM = 1024

# Get text embeddings via the shared project config
embeddings = create_embeddings(dimensions=EMBED_DIM)

# Embedding generation function
def embed(texts: list[str]) -> list[list[float]]:
    return embeddings.embed_documents(texts)

# Smoke test for text embeddings
texts = [
    "LangGraph middleware is powerful",
    "LangGraph MCP is useful",
]
vectors = embed(texts)

len(vectors), len(vectors[0])


### 2）读写长期记忆

先向 InMemoryStore 中写入两条数据。

In [ ]:
# 创建 InMemoryStore 内存存储
store = InMemoryStore(index={"embed": embed, "dims": EMBED_DIM})

# 添加两条用户数据 user_1 user_2
namespace = ("users", )

store.put(
    namespace,  # Namespace to group related data together
    "user_1",  # Key within the namespace
    {
        "rules": [
            "User likes short, direct language",
            "User only speaks English & python",
        ],
        "rule_id": "3",
    },
)

store.put(
    ("users",),
    "user_2",
    {
        "name": "John Smith",
        "language": "English",
    }
)


通过 namespace 和 key，可以直接读取长期记忆。

In [ ]:
item = store.get(namespace, "user_2")
item


也可以通过向量检索召回。

In [ ]:
items = store.search( 
    namespace,
    query="language preferences",
    filter={"rule_id": "3"},
)
items


### 3）使用工具读取长期记忆

In [ ]:
@dataclass
class Context:
    user_id: str

@tool
def get_user_info(runtime: ToolRuntime[Context]) -> str:
    """用于查询用户信息"""
    user_id = runtime.context.user_id
    user_info = runtime.store.get(("users",), user_id) 
    return str(user_info.value) if user_info else "未知用户"

# 创建Agent
agent = create_agent(
    model=model,
    tools=[get_user_info],
    store=store, 
    context_schema=Context
)

# 运行Agent
result = agent.invoke(
    {"messages": [{"role": "user", "content": "查阅用户信息"}]},
    context=Context(user_id="user_2") 
)

for message in result['messages']:
    message.pretty_print()


### 4）使用工具写入长期记忆

In [ ]:
class UserInfo(TypedDict):
    name: str

@tool
def save_user_info(user_info: UserInfo, runtime: ToolRuntime[Context]) -> str:
    """用于保存/更新用户信息"""
    user_id = runtime.context.user_id
    runtime.store.put(("users",), user_id, user_info) 
    return "成功保存用户信息"

# 创建gent
agent = create_agent(
    model=model,
    tools=[save_user_info],
    store=store,
    context_schema=Context
)

# 运行Agent
agent.invoke(
    {"messages": [{"role": "user", "content": "My name is John Smith"}]},
    context=Context(user_id="user_123") 
)

store.get(("users",), "user_123").value
